<a href="https://colab.research.google.com/github/talhanoor23/my-projects/blob/main/Genetic-Mutation-Detection-System/01_Data_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get update
!apt-get install -y samtools

In [ ]:
!wget https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz
!gunzip variant_summary.txt.gz

In [ ]:
!wget https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz
!gunzip hg38.fa.gz
!samtools faidx hg38.fa

In [ ]:
import pandas as pd

df = pd.read_csv("variant_summary.txt", sep="\t", low_memory=False)

In [ ]:
df

,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),...,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification
0,15041,Indel,NM_014855.3(AP5Z1):c.80_83delinsTGCTGTAAACTGTA...,9907,AP5Z1,HGNC:22197,Pathogenic/Likely pathogenic,1,"Dec 17, 2024",397704705,...,TGCTGTAAACTGTAACTGTAAA,-,-,-,-,-,-,SCV001451119|SCV005622007|SCV005909190,-,-
1,15041,Indel,NM_014855.3(AP5Z1):c.80_83delinsTGCTGTAAACTGTA...,9907,AP5Z1,HGNC:22197,Pathogenic/Likely pathogenic,1,"Dec 17, 2024",397704705,...,TGCTGTAAACTGTAACTGTAAA,-,-,-,-,-,-,SCV001451119|SCV005622007|SCV005909190,-,-
2,15042,Deletion,NM_014855.3(AP5Z1):c.1413_1426del (p.Leu473fs),9907,AP5Z1,HGNC:22197,Pathogenic,1,"Jun 29, 2010",397704709,...,G,-,-,-,-,-,-,SCV000020156,-,-
3,15042,Deletion,NM_014855.3(AP5Z1):c.1413_1426del (p.Leu473fs),9907,AP5Z1,HGNC:22197,Pathogenic,1,"Jun 29, 2010",397704709,...,G,-,-,-,-,-,-,SCV000020156,-,-
4,15043,single nucleotide variant,NM_014630.3(ZNF592):c.3136G>A (p.Gly1046Arg),9640,ZNF592,HGNC:28986,Uncertain significance,0,"Jun 29, 2015",150829393,...,A,-,-,-,-,-,-,SCV000020157,-,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6866039,3894784,single nucleotide variant,NM_017617.5(NOTCH1):c.1193G>A (p.Cys398Tyr),4851,NOTCH1,HGNC:7881,Uncertain significance,0,"Apr 01, 2024",-1,...,T,-,-,-,-,-,-,SCV005911581,-,-
6866040,3894785,single nucleotide variant,NM_001356.5(DDX3X):c.1526A>T (p.Asn509Ile),1654,DDX3X,HGNC:2745,Likely pathogenic,1,"Apr 01, 2024",-1,...,T,-,-,-,-,-,-,SCV005911582,-,-
6866041,3894785,single nucleotide variant,NM_001356.5(DDX3X):c.1526A>T (p.Asn509Ile),1654,DDX3X,HGNC:2745,Likely pathogenic,1,"Apr 01, 2024",-1,...,T,-,-,-,-,-,-,SCV005911582,-,-
6866042,3894786,Duplication,NM_015335.5(MED13L):c.3191dup (p.Thr1065fs),23389,MED13L,HGNC:22962,Pathogenic,1,"Apr 01, 2024",-1,...,GC,-,-,-,-,-,-,SCV005911583,-,-


In [ ]:
df.info()

In [ ]:
df["Assembly"].unique()

array(['GRCh37', 'GRCh38', 'na', 'NCBI36'], dtype=object)

In [ ]:
df.shape

(6866044, 43)

In [ ]:
df["AlternateAlleleVCF"].unique()

array(['TGCTGTAAACTGTAACTGTAAA', 'G', 'A', ..., 'AGCGCAGCATCATG',
       'CTATAGT', 'GCCGCCAGCCGCGGCCA'], dtype=object)

In [ ]:
df["ReferenceAlleleVCF"].unique()

In [ ]:
df_hg19 = df[df["Assembly"] == 'GRCh37']
df_hg38 = df[df["Assembly"] == 'GRCh38']
df_unknown = df[df["Assembly"].isin(['na', 'NCBI36'])]

In [ ]:
df_hg38.shape

(3401245, 43)

In [ ]:
df_hg19.shape

(3452941, 43)

In [ ]:
print("Before cleaning df_hg38:", df_hg38.shape)

Before cleaning df_hg38: (3401245, 43)


In [ ]:
df_hg38["ReferenceAlleleVCF"].unique()

array(['GGAT', 'GCTGCTGGACCTGCC', 'G', ..., 'ACCCGCAGGAGTTCATGAAGGCC',
       'GCTCTGTCCACAGGGATGCTGCCTGACCCCAAGAA',
       'TCCGGAAAAACTTTTTTTAAAAAATTAGCCGGGTGTGGTGGTGCATGCCCGTGGTGCCAGCTACTTAGAGGGCTGAGGTGGGATGACCCGTAAGCCCCGGAGGTCGAGGCTGCAGTGAGCCGTGATCGCGTCACTGCACTCAGACTGGACGACAGCGCGAGACCTGTCTCAAAAAAAAATAACAAGGGGTAGGGTGGGGCAGTGAGATGGTTCAGCGTTGTTCCCAAGGAACTCATCTCCCTAAATTTCCAGTCTCAAGCGCCGCTTTCATTCCCTGCCCCGAGCGGCGCTCCCCGCTCCGTGCACGCATCTCCGCTGAGGGCCGCCCTGACGCACGGCCTTCTGCCTTGAACACAGAACCCCTCAGCCCTTCCTATTAGCCGTTTCTGGCCGGGACTGCACGCGCTCCTTCCTGCCTCGGGGCGCAGGGTGAGACTTACGTCTAAAGGGCGCCCGGCGAGCTCGCAGCAGCCCCTGCCCCGCGGAGGGCCGGTGGCGCGCGGCGTCCGGTGAGGGGCAGCGGCCCGGGCTCGGTTCCGGCTTCCAGCAGCGGCCCGAAGGCGCCGGAGCACGCTCCCCGCTCCCCAGAACGCAGCTTCCGCGGCTGCAGCCCAGGCAGCTCTTCGGGAAGCGTCCGCCCGTGGTCTCGCCAGGGGGTTTCCGACCTCCCCGCCCGACAGCGCAGGGGCGGGGGCCGCGGCCGGAGAGCTCGCCCGAGAGCACCCCCTCCCGCTGCGCCTCTGGGCGCACGCGCAGCCGCTGCAGCCTCCCTTATTTAGTCCGCGATGGCTTCCCTCGCGCCCCACCGTCCTCTTCCGGAAGGCGGCTCCCTCCCTGCGCAGCCCGGAGCCCCTGAGATCAGCCTCGAGCAGGCG

In [ ]:
valid_bases = ['A', 'T', 'C', 'G']

df_hg38 = df_hg38[
    df["ReferenceAlleleVCF"].isin(valid_bases) &
    df["AlternateAlleleVCF"].isin(valid_bases)
]

<ipython-input-16-f6acce0d505a>:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_hg38 = df_hg38[


In [ ]:
print("After cleaning df_hg38:", df_hg38.shape)

After cleaning df_hg38: (3115046, 43)


In [ ]:
df_hg38["AlternateAlleleVCF"].unique()

array(['A', 'T', 'G', 'C'], dtype=object)

In [ ]:
df_hg38["ReferenceAlleleVCF"].unique()

array(['G', 'C', 'A', 'T'], dtype=object)

In [ ]:
df_hg38['AlternateAllele'] = df_hg38['AlternateAlleleVCF'].str.upper()
df_hg38['ReferenceAllele'] = df_hg38['ReferenceAlleleVCF'].str.upper()

In [ ]:
df_hg38.drop(columns=['ReferenceAlleleVCF', 'AlternateAlleleVCF'], inplace=True)

In [ ]:
df_hg38

In [ ]:
df_hg38['ClinicalSignificance'].unique()

In [ ]:
df_hg38['Type'].unique()

array(['single nucleotide variant', 'Variation', 'Deletion',
       'Duplication'], dtype=object)

In [ ]:
df_hg38 = df_hg38[df_hg38['Type'] == 'single nucleotide variant']

In [ ]:
df_hg38 = df_hg38[df_hg38['ClinicalSignificance'].isin(['Pathogenic', 'Benign'])]

In [ ]:
df_hg38['label'] = df_hg38['ClinicalSignificance'].map({'Pathogenic': 1, 'Benign': 0})

<ipython-input-27-2b70a70abeb7>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_hg38['label'] = df_hg38['ClinicalSignificance'].map({'Pathogenic': 1, 'Benign': 0})


In [ ]:
df_filtered = df_hg38[['Chromosome', 'Start', 'ReferenceAllele', 'AlternateAllele', 'GeneSymbol', 'label']]

In [ ]:
df_filtered.to_csv("clinvar_snvs_filtered.csv", index=False)

In [ ]:
df_filtered

,Chromosome,Start,ReferenceAllele,AlternateAllele,GeneSymbol,label
7,11,126275389,C,T,FOXRED1,1
23,6,26093008,G,A,HFE,0
31,6,26093215,G,T,HFE,1
35,2,19989284,T,C,WDR35,1
37,2,19945787,T,C,WDR35,1
...,...,...,...,...,...,...
6865973,16,29990867,C,A,TAOK2,1
6866015,8,15659648,G,C,TUSC3,1
6866017,X,77684291,G,C,ATRX,1
6866019,8,60856514,G,T,CHD7,1


In [ ]:
!pip install pyfaidx

In [ ]:
from pyfaidx import Fasta
import pandas as pd

# Load the reference genome
genome = Fasta("hg38.fa")

# Load your mutation dataset
df = pd.read_csv("clinvar_snvs_filtered.csv")
print(df.head())

  Chromosome      Start ReferenceAllele AlternateAllele GeneSymbol  label
0         11  126275389               C               T    FOXRED1      1
1          6   26093008               G               A        HFE      0
2          6   26093215               G               T        HFE      1
3          2   19989284               T               C      WDR35      1
4          2   19945787               T               C      WDR35      1


In [ ]:
# Check which chromosomes from df_clean are NOT in genome
df['Chromosome'] = 'chr' + df['Chromosome'].astype(str)
missing_chroms = set(df['Chromosome']) - set(genome.keys())
print("Missing Chromosomes:", missing_chroms)

Missing Chromosomes: {'chrMT'}


In [ ]:
df.head()

,Chromosome,Start,ReferenceAllele,AlternateAllele,GeneSymbol,label
0,chr11,126275389,C,T,FOXRED1,1
1,chr6,26093008,G,A,HFE,0
2,chr6,26093215,G,T,HFE,1
3,chr2,19989284,T,C,WDR35,1
4,chr2,19945787,T,C,WDR35,1


In [ ]:
df["Chromosome"].unique()

array(['chr11', 'chr6', 'chr2', 'chr20', 'chr10', 'chr19', 'chr16',
       'chr22', 'chr1', 'chr8', 'chr14', 'chr21', 'chr5', 'chr4', 'chr18',
       'chr15', 'chr3', 'chr17', 'chr12', 'chr7', 'chr9', 'chr13',
       'chrMT', 'chrY', 'chrX'], dtype=object)

In [ ]:
genome.keys()

odict_keys(['chr1', 'chr10', 'chr11', 'chr11_KI270721v1_random', 'chr12', 'chr13', 'chr14', 'chr14_GL000009v2_random', 'chr14_GL000225v1_random', 'chr14_KI270722v1_random', 'chr14_GL000194v1_random', 'chr14_KI270723v1_random', 'chr14_KI270724v1_random', 'chr14_KI270725v1_random', 'chr14_KI270726v1_random', 'chr15', 'chr15_KI270727v1_random', 'chr16', 'chr16_KI270728v1_random', 'chr17', 'chr17_GL000205v2_random', 'chr17_KI270729v1_random', 'chr17_KI270730v1_random', 'chr18', 'chr19', 'chr1_KI270706v1_random', 'chr1_KI270707v1_random', 'chr1_KI270708v1_random', 'chr1_KI270709v1_random', 'chr1_KI270710v1_random', 'chr1_KI270711v1_random', 'chr1_KI270712v1_random', 'chr1_KI270713v1_random', 'chr1_KI270714v1_random', 'chr2', 'chr20', 'chr21', 'chr22', 'chr22_KI270731v1_random', 'chr22_KI270732v1_random', 'chr22_KI270733v1_random', 'chr22_KI270734v1_random', 'chr22_KI270735v1_random', 'chr22_KI270736v1_random', 'chr22_KI270737v1_random', 'chr22_KI270738v1_random', 'chr22_KI270739v1_random', 

In [ ]:
# Function to test ref match
def check_ref_mismatch(row):
    try:
        chrom = row['Chromosome']
        pos = int(row['Start'])
        ref = row['ReferenceAllele'].upper()
        genome_ref = genome[chrom][pos - 1].seq.upper()
        return genome_ref != ref
    except:
        return True

df['RefMismatch'] = df.apply(check_ref_mismatch, axis=1)
print("Ref mismatches:", df['RefMismatch'].sum())

Ref mismatches: 1007


In [ ]:
# Fix chromosome names properly
df['Chromosome'] = df['Chromosome'].apply(lambda x: x if str(x).startswith('chr') else f'chr{x}')

In [ ]:
df["Chromosome"].unique()

array(['chr11', 'chr6', 'chr2', 'chr20', 'chr10', 'chr19', 'chr16',
       'chr22', 'chr1', 'chr8', 'chr14', 'chr21', 'chr5', 'chr4', 'chr18',
       'chr15', 'chr3', 'chr17', 'chr12', 'chr7', 'chr9', 'chr13',
       'chrMT', 'chrY', 'chrX'], dtype=object)

In [ ]:
df

,Chromosome,Start,ReferenceAllele,AlternateAllele,GeneSymbol,label,RefMismatch
0,chr11,126275389,C,T,FOXRED1,1,False
1,chr6,26093008,G,A,HFE,0,False
2,chr6,26093215,G,T,HFE,1,False
3,chr2,19989284,T,C,WDR35,1,False
4,chr2,19945787,T,C,WDR35,1,False
...,...,...,...,...,...,...,...
261578,chr16,29990867,C,A,TAOK2,1,False
261579,chr8,15659648,G,C,TUSC3,1,False
261580,chrX,77684291,G,C,ATRX,1,False
261581,chr8,60856514,G,T,CHD7,1,False


In [ ]:
df['RefMismatch'].value_counts()

,count
RefMismatch,
False,260576
True,1007


In [ ]:
def is_valid_ref(row):
    try:
        genome_base = genome[row['Chromosome']][int(row['Start']) - 1].seq.upper()
        return genome_base == row['ReferenceAllele'].upper()
    except:
        return False

df_valid = df[df.apply(is_valid_ref, axis=1)].copy()
print(f"Valid entries: {len(df_valid)}")


Valid entries: 260576


In [ ]:
!grep ">" hg38.fa

>chr1
>chr10
>chr11
>chr11_KI270721v1_random
>chr12
>chr13
>chr14
>chr14_GL000009v2_random
>chr14_GL000225v1_random
>chr14_KI270722v1_random
>chr14_GL000194v1_random
>chr14_KI270723v1_random
>chr14_KI270724v1_random
>chr14_KI270725v1_random
>chr14_KI270726v1_random
>chr15
>chr15_KI270727v1_random
>chr16
>chr16_KI270728v1_random
>chr17
>chr17_GL000205v2_random
>chr17_KI270729v1_random
>chr17_KI270730v1_random
>chr18
>chr19
>chr1_KI270706v1_random
>chr1_KI270707v1_random
>chr1_KI270708v1_random
>chr1_KI270709v1_random
>chr1_KI270710v1_random
>chr1_KI270711v1_random
>chr1_KI270712v1_random
>chr1_KI270713v1_random
>chr1_KI270714v1_random
>chr2
>chr20
>chr21
>chr22
>chr22_KI270731v1_random
>chr22_KI270732v1_random
>chr22_KI270733v1_random
>chr22_KI270734v1_random
>chr22_KI270735v1_random
>chr22_KI270736v1_random
>chr22_KI270737v1_random
>chr22_KI270738v1_random
>chr22_KI270739v1_random
>chr2_KI270715v1_random
>chr2_KI270716v1_random
>chr3
>chr3_GL000221v1_random
>chr4
>chr4_GL000008v2_rando

In [ ]:
from pyfaidx import Fasta
import pandas as pd

# Load the hg38 genome (make sure .fai is generated or pyfaidx will do it)
genome = Fasta("hg38.fa")

# Ensure chromosome naming matches (e.g., '11' -> 'chr11')
# df['Chromosome'] = 'chr' + df['Chromosome'].astype(str)

# Function to get 101-bp context around the mutation site
def get_sequence(chrom, pos, ref, alt):
    try:
        # Adjust to 0-based indexing
        start = max(pos - 51, 0)
        # start = pos - 51
        end = pos + 50

        seq = genome[chrom][start:end].seq

        # Confirm the reference base matches
        ref_base = genome[chrom][pos - 1].seq.upper()
        if ref_base != ref:
            return None  # Skip if reference base does not match
        return seq.upper()
    except KeyError:
        return None  # Chromosome not found
    except Exception as e:
        return None  # Other errors

# Apply to DataFrame
df['Sequence'] = df.apply(
    lambda row: get_sequence(row['Chromosome'], int(row['Start']), row['ReferenceAllele'], row['AlternateAllele']),
    axis=1
)

# Drop rows where sequence couldn't be retrieved
df_clean = df.dropna(subset=['Sequence'])

df_clean.head()


,Chromosome,Start,ReferenceAllele,AlternateAllele,GeneSymbol,label,RefMismatch,Sequence
0,chr11,126275389,C,T,FOXRED1,1,False,AAGGTTGGTTTGACCCCTGGTGTCTGCTCCAGGGGCTTCGGCGAAA...
1,chr6,26093008,G,A,HFE,0,False,GGGGTATGTGACTGATGAGAGCCAGGAGCTGAGAAAATCTATTGGG...
2,chr6,26093215,G,T,HFE,1,False,TGCTGTTTTTGTCGTCATCTTGTTCATTGGAATTTTGTTCATAATA...
3,chr2,19989284,T,C,WDR35,1,False,CCTTGTTCCAGGATACACACTGCAGCTTCACGTTATTGGGAATGGA...
4,chr2,19945787,T,C,WDR35,1,False,TATTTGGCTCATTATTTATAGACTGTTAACACTCATTTCTTGTTTT...


In [ ]:
df_clean['RefMismatch'].value_counts()

,count
RefMismatch,
False,260576


In [ ]:
df_clean['Sequence'][1]

'GGGGTATGTGACTGATGAGAGCCAGGAGCTGAGAAAATCTATTGGGGGTTGAGAGGAGTGCCTGAGGAGGTAATTATGGCAGTGAGATGAGGATCTGCTCT'

In [ ]:
df_clean['seq_length'] = df_clean['Sequence'].apply(len)
print(df_clean['seq_length'].value_counts())

seq_length
101    260576
Name: count, dtype: int64


<ipython-input-60-9be09dc0d18f>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['seq_length'] = df_clean['Sequence'].apply(len)


In [ ]:
df_clean['Chromosome'].unique()

array(['chr11', 'chr6', 'chr2', 'chr20', 'chr10', 'chr19', 'chr16',
       'chr22', 'chr1', 'chr8', 'chr14', 'chr21', 'chr5', 'chr4', 'chr18',
       'chr15', 'chr3', 'chr17', 'chr12', 'chr7', 'chr9', 'chr13', 'chrY',
       'chrX'], dtype=object)

In [ ]:
df_clean.shape

(260576, 9)

In [ ]:
df_clean

,Chromosome,Start,ReferenceAllele,AlternateAllele,GeneSymbol,label,RefMismatch,Sequence,seq_length
0,chr11,126275389,C,T,FOXRED1,1,False,AAGGTTGGTTTGACCCCTGGTGTCTGCTCCAGGGGCTTCGGCGAAA...,101
1,chr6,26093008,G,A,HFE,0,False,GGGGTATGTGACTGATGAGAGCCAGGAGCTGAGAAAATCTATTGGG...,101
2,chr6,26093215,G,T,HFE,1,False,TGCTGTTTTTGTCGTCATCTTGTTCATTGGAATTTTGTTCATAATA...,101
3,chr2,19989284,T,C,WDR35,1,False,CCTTGTTCCAGGATACACACTGCAGCTTCACGTTATTGGGAATGGA...,101
4,chr2,19945787,T,C,WDR35,1,False,TATTTGGCTCATTATTTATAGACTGTTAACACTCATTTCTTGTTTT...,101
...,...,...,...,...,...,...,...,...,...
261578,chr16,29990867,C,A,TAOK2,1,False,CAGGCCCTTCGGCAGCAGCTTCAACAGGAGCTGGAGCTGCTCAACG...,101
261579,chr8,15659648,G,C,TUSC3,1,False,TTGCAGCTGAGCAACTAGCAAAGTGGATTGCTGACAGAACGGATGT...,101
261580,chrX,77684291,G,C,ATRX,1,False,CAGAGCCAGAACAGGAATCATCTAATTTCTTTTCTTCTCCATTACA...,101
261581,chr8,60856514,G,T,CHD7,1,False,CTGTCCCTCGCCAGCGGAGGAGGAGGAGGAGAAAAATCGAAATTGA...,101


In [ ]:
df_clean.to_csv("mutation_sequences.csv", index=False)
print("Saved df_clean as mutation_sequences.csv")

Saved df_clean as mutation_sequences.csv


In [ ]:
import pandas as pd

# Tokenization function
def kmer_tokenizer(seq, k=6):
    return ' '.join([seq[i:i+k] for i in range(len(seq)-k+1)])

# Load the saved sequences
df = pd.read_csv("mutation_sequences.csv")

# Tokenize sequences
df['tokenized'] = df['Sequence'].apply(lambda x: kmer_tokenizer(x, k=6))

# Save the tokenized data
df.to_csv("mutation_sequences_kmer.csv", index=False)
print("Tokenized dataset saved as mutation_sequences_kmer.csv")



Tokenized dataset saved as mutation_sequences_kmer.csv


In [ ]:
df['tokenized'][1]

'GGGGTA GGGTAT GGTATG GTATGT TATGTG ATGTGA TGTGAC GTGACT TGACTG GACTGA ACTGAT CTGATG TGATGA GATGAG ATGAGA TGAGAG GAGAGC AGAGCC GAGCCA AGCCAG GCCAGG CCAGGA CAGGAG AGGAGC GGAGCT GAGCTG AGCTGA GCTGAG CTGAGA TGAGAA GAGAAA AGAAAA GAAAAT AAAATC AAATCT AATCTA ATCTAT TCTATT CTATTG TATTGG ATTGGG TTGGGG TGGGGG GGGGGT GGGGTT GGGTTG GGTTGA GTTGAG TTGAGA TGAGAG GAGAGG AGAGGA GAGGAG AGGAGT GGAGTG GAGTGC AGTGCC GTGCCT TGCCTG GCCTGA CCTGAG CTGAGG TGAGGA GAGGAG AGGAGG GGAGGT GAGGTA AGGTAA GGTAAT GTAATT TAATTA AATTAT ATTATG TTATGG TATGGC ATGGCA TGGCAG GGCAGT GCAGTG CAGTGA AGTGAG GTGAGA TGAGAT GAGATG AGATGA GATGAG ATGAGG TGAGGA GAGGAT AGGATC GGATCT GATCTG ATCTGC TCTGCT CTGCTC TGCTCT'

In [ ]:
df['seq_length'] = df['tokenized'].apply(len)
print(df['seq_length'].value_counts())

seq_length
671    260576
Name: count, dtype: int64


In [ ]:
df